# Phase 1 — Final experiments (Remote Sensing Letters submission)

Every model is trained and evaluated under **one protocol**, identical to the SNSM `6_dataset` notebook:

| Step | Setting |
|---|---|
| Preprocessing | Wiener 5×5 per band → PCA(30) fitted on the whole scene → + db4 LL wavelet sub-band = 60 features |
| Sampling | 15×15 patches centred on labelled pixels, 5% per-class training split, rest is test (capped at 50,000 for WHU-Hi) |
| Seeds | 42, 43, 44, 45, 46 (same splits as the existing SNSM runs) |
| Training | AdamW (lr 1e-3, wd 1e-4), StepLR(40, 0.5), **100 fixed epochs**, batch 64 |
| Selection | **None** — no validation/test-based checkpointing or early stopping; the final-epoch model is evaluated once |

**How to run on Kaggle:** add the `datasets_hsi_all` dataset as input, set Accelerator = GPU T4, then *Save Version → Save & Run All*.
Everything is written to `/kaggle/working/final/` (download it and put it in `code_and_results/final/`).
If the session dies, re-running in the same session skips runs already in `runs.csv`.

In [ ]:
# =============================================================================
# Cell 1: Setup & configuration
# =============================================================================
import os, sys, time, json, glob, copy, random, subprocess, gc

try:
    import thop
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop'], check=False)
    try:
        import thop
    except ImportError:
        thop = None

import numpy as np
import scipy.io as sio
from scipy import signal
from scipy.ndimage import zoom
import pywt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

SMOKE = os.environ.get('PHASE1_SMOKE') == '1'   # local synthetic-data test only

# ── Protocol (do not change between models) ──
SEEDS          = [42, 43, 44, 45, 46]
EPOCHS         = 100
BATCH_SIZE     = 64
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
STEP_SIZE      = 40
STEP_GAMMA     = 0.5
WINDOW         = 15
TRAIN_RATIO    = 0.05
PCA_COMPONENTS = 30
MAX_TEST       = 50000

# ── What to run ──
CORE_DATASETS  = ['IndianPines', 'PaviaU', 'KSC', 'Botswana']
LARGE_DATASETS = ['Salinas', 'WHU_Hi_HanChuan']      # SNSM only (near-saturated)
RUN_LARGE      = True
CORE_MODELS    = ['plain', 'se_gamma', 'eca_gamma', 'cbam', 'fc_gamma_beta', 'snsm_gamma', 'snsm', 'hybridsn']
SSFTT_DATASETS = ['IndianPines', 'KSC']
MAP_DATASET, MAP_MODEL, MAP_SEED = 'IndianPines', 'snsm', 42

OUT_DIR = '/kaggle/working/final'
if SMOKE:
    SEEDS, EPOCHS, MAX_TEST = [42, 43], 2, 300
    OUT_DIR = os.environ['PHASE1_OUT']
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env = {'torch': torch.__version__, 'device': str(device),
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'thop': thop is not None, 'smoke': SMOKE}
with open(os.path.join(OUT_DIR, 'env.json'), 'w') as f:
    json.dump(env, f, indent=2)
print(env)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# =============================================================================
# Cell 2: Data loading, preprocessing and splits
# =============================================================================
DATASET_INFO = {
    'IndianPines':     {'data_key': 'indian_pines_corrected', 'gt_key': 'indian_pines_gt'},
    'PaviaU':          {'data_key': 'paviaU',                 'gt_key': 'paviaU_gt'},
    'Salinas':         {'data_key': 'salinas_corrected',      'gt_key': 'salinas_gt'},
    'Botswana':        {'data_key': 'Botswana',               'gt_key': 'Botswana_gt'},
    'KSC':             {'data_key': 'KSC',                    'gt_key': 'KSC_gt'},
    'WHU_Hi_HanChuan': {'data_key': 'WHU_Hi_HanChuan',        'gt_key': 'WHU_Hi_HanChuan_gt'},
}
CLASS_NAMES = {
    'IndianPines': ['Alfalfa', 'Corn-notill', 'Corn-mintill', 'Corn', 'Grass-pasture',
                    'Grass-trees', 'Grass-pasture-mowed', 'Hay-windrowed', 'Oats',
                    'Soybean-notill', 'Soybean-mintill', 'Soybean-clean', 'Wheat', 'Woods',
                    'Bldg-Grass-Trees-Drives', 'Stone-Steel-Towers'],
    'PaviaU': ['Asphalt', 'Meadows', 'Gravel', 'Trees', 'Painted metal sheets', 'Bare Soil',
               'Bitumen', 'Self-Blocking Bricks', 'Shadows'],
    'KSC': ['Scrub', 'Willow swamp', 'Cabbage palm hammock', 'Cabbage palm/oak hammock',
            'Slash pine', 'Oak/broadleaf hammock', 'Hardwood swamp', 'Graminoid marsh',
            'Spartina marsh', 'Cattail marsh', 'Salt marsh', 'Mud flats', 'Water'],
    'Botswana': ['Water', 'Hippo grass', 'Floodplain grasses 1', 'Floodplain grasses 2',
                 'Reeds', 'Riparian', 'Firescar', 'Island interior', 'Acacia woodlands',
                 'Acacia shrublands', 'Acacia grasslands', 'Short mopane', 'Mixed mopane',
                 'Exposed soils'],
}
DATA_DIRS = ['/kaggle/input/datasets/devgurucodes/datasets-hsi-all',
             '/kaggle/input/datasets-hsi-all']


def _find(fname):
    for d in DATA_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(f'{fname} not found under /kaggle/input — add the datasets_hsi_all input')


def _synthetic(name):
    # Small fake scene with class-specific spectra (local smoke test only).
    rng = np.random.RandomState(sum(map(ord, name)))
    h, w, b, k = 40, 36, 48, 5
    gt = np.zeros((h, w), dtype=np.int32)
    for c in range(1, k + 1):
        r0, c0 = rng.randint(0, h - 10), rng.randint(0, w - 10)
        gt[r0:r0 + 10, c0:c0 + 10] = c
    sig = rng.rand(k + 1, b) * 100
    img = sig[gt] + rng.randn(h, w, b) * 5
    return img.astype(np.float32), gt


def load_dataset(name):
    if SMOKE:
        return _synthetic(name)
    info = DATASET_INFO[name]
    img = sio.loadmat(_find(f'{name}_data.mat'))[info['data_key']]
    gt = sio.loadmat(_find(f'{name}_gt.mat'))[info['gt_key']]
    return img, gt


def wiener_denoise(X):
    out = np.zeros(X.shape, dtype=np.float32)
    for b in range(X.shape[2]):
        out[:, :, b] = signal.wiener(X[:, :, b].astype(np.float64), mysize=(5, 5)).astype(np.float32)
    return out


def apply_pca(X, n):
    h, w, b = X.shape
    return PCA(n_components=n).fit_transform(X.reshape(-1, b)).reshape(h, w, n).astype(np.float32)


def wavelet_features(X):
    h, w, b = X.shape
    ll = np.zeros((h, w, b), dtype=np.float32)
    for i in range(b):
        LL, _ = pywt.dwt2(X[:, :, i].astype(np.float64), 'db4')
        ll[:, :, i] = zoom(LL, (h / LL.shape[0], w / LL.shape[1]), order=1)[:h, :w]
    return np.concatenate([X, ll], axis=2)


def preprocess(img):
    return wavelet_features(apply_pca(wiener_denoise(img), PCA_COMPONENTS))


def split_centers(gt, seed):
    # Same per-class shuffling as the SNSM notebook, so seeds 42-46 give identical splits.
    rng = np.random.RandomState(seed)
    tr_xy, tr_y, te_xy, te_y = [], [], [], []
    for c in np.unique(gt[gt > 0]):
        idx = np.argwhere(gt == c)
        rng.shuffle(idx)
        n = max(1, int(len(idx) * TRAIN_RATIO))
        tr_xy.append(idx[:n]); tr_y.append(np.full(n, c - 1))
        te_xy.append(idx[n:]); te_y.append(np.full(len(idx) - n, c - 1))
    tr_xy, tr_y = np.concatenate(tr_xy), np.concatenate(tr_y)
    te_xy, te_y = np.concatenate(te_xy), np.concatenate(te_y)
    if len(te_y) > MAX_TEST:
        keep = np.sort(np.random.RandomState(seed).choice(len(te_y), MAX_TEST, replace=False))
        te_xy, te_y = te_xy[keep], te_y[keep]
    return tr_xy, tr_y.astype(np.int64), te_xy, te_y.astype(np.int64)


def pad(X):
    m = WINDOW // 2
    return np.pad(X, ((m, m), (m, m), (0, 0))).astype(np.float32)


def extract_patches(padded, coords):
    out = np.empty((len(coords), padded.shape[2], WINDOW, WINDOW), dtype=np.float32)
    for i, (r, c) in enumerate(coords):
        out[i] = padded[r:r + WINDOW, c:c + WINDOW, :].transpose(2, 0, 1)
    return out

print('Data utilities ready.')

In [ ]:
# =============================================================================
# Cell 3: Models
# =============================================================================
# All modulation variants share one backbone (identical to SNSM_CNN):
#   conv3x3(bands->64)+BN+ReLU -> 2 residual blocks -> GAP -> dropout 0.4 -> FC
# Each residual block: conv-BN-ReLU -> [modulation] -> conv-BN -> +x -> ReLU
# Variants differ ONLY in the modulation step:
#   plain          : none
#   se_gamma       : SE-style, GAP -> FC -> sigmoid scale (gamma only)
#   eca_gamma      : ECA, GAP -> Conv1d(k=3) across channels -> sigmoid scale (gamma only)
#   cbam           : CBAM, channel gate (shared MLP on avg+max pool) then 7x7 spatial gate
#   fc_gamma_beta  : GAP -> FC -> sigmoid scale + linear shift (previous SSMRB)
#   snsm_gamma     : spectral-neighbourhood descriptors -> Conv1d -> scale only
#   snsm           : spectral-neighbourhood descriptors -> Conv1d -> scale + shift (proposed)

class SEGamma(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(ch, reduction)
        self.fc_gamma = nn.Linear(reduction, ch)

    def forward(self, x):
        z = F.relu(self.fc1(x.mean(dim=(2, 3))))
        return x * torch.sigmoid(self.fc_gamma(z))[:, :, None, None]


class ECAGamma(nn.Module):
    # Wang et al. (2020); adaptive kernel size for C=64 is 3.
    def __init__(self, ch, k=3):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        z = self.conv(x.mean(dim=(2, 3)).unsqueeze(1)).squeeze(1)
        return x * torch.sigmoid(z)[:, :, None, None]


class CBAM(nn.Module):
    # Woo et al. (2018): channel attention (reduction 16) followed by 7x7 spatial attention.
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(ch, ch // reduction), nn.ReLU(), nn.Linear(ch // reduction, ch))
        self.spatial = nn.Conv2d(2, 1, 7, padding=3, bias=False)

    def forward(self, x):
        gate = torch.sigmoid(self.mlp(x.mean(dim=(2, 3))) + self.mlp(x.amax(dim=(2, 3))))
        x = x * gate[:, :, None, None]
        s = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * torch.sigmoid(self.spatial(s))


class FCGammaBeta(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(ch, reduction)
        self.fc_gamma = nn.Linear(reduction, ch)
        self.fc_beta = nn.Linear(reduction, ch)

    def forward(self, x):
        z = F.relu(self.fc1(x.mean(dim=(2, 3))))
        gamma = torch.sigmoid(self.fc_gamma(z))[:, :, None, None]
        beta = self.fc_beta(z)[:, :, None, None]
        return x * gamma + beta


class SNSMModulation(nn.Module):
    # Spectral-neighbourhood self-modulation (mean/var/gradient descriptors, hidden=8, normalised).
    def __init__(self, ch, hidden=8, use_beta=True):
        super().__init__()
        self.use_beta = use_beta
        self.spectral_conv1 = nn.Conv1d(3, hidden, kernel_size=3, padding=2, dilation=2,
                                        padding_mode='replicate')
        self.spectral_conv2 = nn.Conv1d(hidden, 2 if use_beta else 1, kernel_size=3, padding=1,
                                        padding_mode='replicate')
        nn.init.kaiming_normal_(self.spectral_conv1.weight, mode='fan_out', nonlinearity='relu')
        nn.init.zeros_(self.spectral_conv2.weight)
        nn.init.zeros_(self.spectral_conv2.bias)

    @staticmethod
    def _norm(s):
        return (s - s.mean(dim=1, keepdim=True)) / (s.std(dim=1, keepdim=True) + 1e-6)

    def forward(self, x):
        mean = x.mean(dim=(2, 3))
        var = x.var(dim=(2, 3), unbiased=False)
        grad = torch.zeros_like(mean)
        grad[:, 1:-1] = (mean[:, 2:] - mean[:, :-2]) / 2.0
        grad[:, 0] = mean[:, 1] - mean[:, 0]
        grad[:, -1] = mean[:, -1] - mean[:, -2]
        stats = torch.stack([self._norm(mean), self._norm(var), self._norm(grad)], dim=1)
        z = self.spectral_conv2(F.relu(self.spectral_conv1(stats)))
        gamma = (1.0 + 0.5 * torch.tanh(z[:, 0]))[:, :, None, None]
        if not self.use_beta:
            return x * gamma
        return x * gamma + z[:, 1][:, :, None, None]


def make_modulation(kind, ch):
    return {'plain': lambda: None,
            'se_gamma': lambda: SEGamma(ch),
            'eca_gamma': lambda: ECAGamma(ch),
            'cbam': lambda: CBAM(ch),
            'fc_gamma_beta': lambda: FCGammaBeta(ch),
            'snsm_gamma': lambda: SNSMModulation(ch, use_beta=False),
            'snsm': lambda: SNSMModulation(ch, use_beta=True)}[kind]()


class ResidualBlock(nn.Module):
    def __init__(self, ch, modulation):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(ch)
        self.mod = modulation
        nn.init.kaiming_normal_(self.conv1.weight, mode='fan_out', nonlinearity='relu')
        nn.init.kaiming_normal_(self.conv2.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        if self.mod is not None:
            out = self.mod(out)
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)


class ModulatedCNN(nn.Module):
    def __init__(self, num_classes, num_bands, kind):
        super().__init__()
        self.conv1 = nn.Conv2d(num_bands, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.block1 = ResidualBlock(64, make_modulation(kind, 64))
        self.block2 = ResidualBlock(64, make_modulation(kind, 64))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(64, num_classes)

    def features(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.block2(self.block1(x))
        return x.mean(dim=(2, 3))

    def classify(self, f):
        return self.fc(self.dropout(f))

    def forward(self, x):
        return self.classify(self.features(x))


class HybridSN(nn.Module):
    # Roy et al. (2020), original layer configuration (valid 3D convs -> 2D conv -> FC 256/128).
    def __init__(self, num_classes, num_bands):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 8, (7, 3, 3)), nn.ReLU(),
            nn.Conv3d(8, 16, (5, 3, 3)), nn.ReLU(),
            nn.Conv3d(16, 32, (3, 3, 3)), nn.ReLU())
        d, s = num_bands - 12, WINDOW - 6
        self.conv2d = nn.Sequential(nn.Conv2d(32 * d, 64, 3), nn.ReLU())
        self.fc = nn.Sequential(
            nn.Linear(64 * (s - 2) ** 2, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.4))
        self.out = nn.Linear(128, num_classes)

    def features(self, x):
        x = self.conv3d(x.unsqueeze(1))
        x = self.conv2d(x.flatten(1, 2))
        return self.fc(x.flatten(1))

    def classify(self, f):
        return self.out(f)

    def forward(self, x):
        return self.classify(self.features(x))


class _Attention(nn.Module):
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.heads, self.scale = heads, (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        b, n, d = x.shape
        q, k, v = self.qkv(x).reshape(b, n, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        attn = ((q @ k.transpose(-2, -1)) * self.scale).softmax(dim=-1)
        return self.drop(self.proj((attn @ v).transpose(1, 2).reshape(b, n, d)))


class SSFTT(nn.Module):
    # Sun et al. (2022): 3D conv -> 2D conv -> Gaussian-weighted feature tokenizer (4 tokens)
    # -> 1 pre-norm transformer layer (8 heads, MLP dim 8) -> CLS token -> linear.
    def __init__(self, num_classes, num_bands, num_tokens=4, dim=64, heads=8, mlp_dim=8, dropout=0.1):
        super().__init__()
        self.conv3d = nn.Sequential(nn.Conv3d(1, 8, (3, 3, 3)), nn.BatchNorm3d(8), nn.ReLU())
        self.conv2d = nn.Sequential(nn.Conv2d(8 * (num_bands - 2), 64, 3), nn.BatchNorm2d(64), nn.ReLU())
        self.token_wA = nn.Parameter(torch.empty(1, num_tokens, 64))
        self.token_wV = nn.Parameter(torch.empty(1, 64, dim))
        nn.init.xavier_normal_(self.token_wA)
        nn.init.xavier_normal_(self.token_wV)
        self.pos_embedding = nn.Parameter(torch.empty(1, num_tokens + 1, dim))
        nn.init.normal_(self.pos_embedding, std=0.02)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.emb_drop = nn.Dropout(dropout)
        self.norm1, self.attn = nn.LayerNorm(dim), _Attention(dim, heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(mlp_dim, dim), nn.Dropout(dropout))
        self.head = nn.Linear(dim, num_classes)

    def features(self, x):
        x = self.conv3d(x.unsqueeze(1))
        x = self.conv2d(x.flatten(1, 2))
        x = x.flatten(2).transpose(1, 2)                                     # (B, HW, 64)
        A = torch.matmul(x, self.token_wA.transpose(1, 2)).transpose(1, 2)   # (B, L, HW)
        tokens = torch.matmul(A.softmax(dim=-1), torch.matmul(x, self.token_wV))
        x = torch.cat([self.cls_token.expand(x.size(0), -1, -1), tokens], dim=1)
        x = self.emb_drop(x + self.pos_embedding)
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x[:, 0]

    def classify(self, f):
        return self.head(f)

    def forward(self, x):
        return self.classify(self.features(x))


def build_model(name, num_classes, num_bands):
    if name == 'hybridsn':
        return HybridSN(num_classes, num_bands)
    if name == 'ssftt':
        return SSFTT(num_classes, num_bands)
    return ModulatedCNN(num_classes, num_bands, name)

for _m in CORE_MODELS + ['ssftt']:
    _net = build_model(_m, 16, 2 * PCA_COMPONENTS)
    _out = _net(torch.randn(2, 2 * PCA_COMPONENTS, WINDOW, WINDOW))
    print(f'{_m:14s} params={sum(p.numel() for p in _net.parameters()):>9,d} out={tuple(_out.shape)}')

In [ ]:
# =============================================================================
# Cell 4: Training, evaluation and efficiency
# =============================================================================
def train_and_evaluate(model, Xtr, ytr, Xte, yte, num_classes, seed):
    # Fixed-epoch training; the final model is evaluated once on the test set.
    Xtr_t = torch.from_numpy(Xtr).to(device)
    ytr_t = torch.from_numpy(ytr).to(device)
    gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)
    criterion = nn.CrossEntropyLoss()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(ytr_t), generator=gen).to(device)
        for i in range(0, len(perm), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            optimizer.zero_grad()
            loss = criterion(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward()
            optimizer.step()
        scheduler.step()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    train_time = time.time() - t0

    model.eval()
    preds, feats = [], []
    with torch.no_grad():
        for i in range(0, len(Xte), 1024):
            xb = torch.from_numpy(Xte[i:i + 1024]).to(device)
            f = model.features(xb)
            preds.append(model.classify(f).argmax(1).cpu().numpy())
            feats.append(f.cpu().numpy())
    del Xtr_t, ytr_t

    y_pred = np.concatenate(preds)
    cm = confusion_matrix(yte, y_pred, labels=list(range(num_classes)))
    support = cm.sum(axis=1)
    per_class = np.where(support > 0, np.diag(cm) / np.maximum(support, 1), np.nan) * 100
    return {
        'OA': float((y_pred == yte).mean() * 100),
        'AA': float(np.nanmean(per_class)),
        'kappa': float(cohen_kappa_score(yte, y_pred)),
        # Fisher is only comparable between the CNN variants (same 64-d GAP feature space).
        'fisher': fisher_ratio(np.concatenate(feats), yte),
        'train_time_s': train_time,
        'per_class': [None if np.isnan(a) else round(float(a), 4) for a in per_class],
    }


def fisher_ratio(features, labels):
    g = features.mean(axis=0)
    inter = intra = 0.0
    for c in np.unique(labels):
        fc = features[labels == c]
        mc = fc.mean(axis=0)
        inter += len(fc) * np.sum((mc - g) ** 2)
        intra += np.sum((fc - mc) ** 2)
    return float(inter / intra) if intra > 0 else 0.0


def measure_efficiency(model, num_bands):
    model.eval()
    params = sum(p.numel() for p in model.parameters())
    dummy = torch.randn(1, num_bands, WINDOW, WINDOW, device=device)
    macs = None  # thop counts MACs of conv/linear layers only; matmul/mean/var ops are not included
    if thop is not None:
        try:
            macs, _ = thop.profile(copy.deepcopy(model), inputs=(dummy,), verbose=False)
        except Exception as e:
            print(f'    thop failed: {e}')
    times = []
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
        for _ in range(100):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.time()
            model(dummy)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.time() - t0)
    return {'params': int(params), 'macs_convlinear': None if macs is None else float(macs),
            'latency_ms': float(np.mean(times) * 1000)}

print('Training / evaluation utilities ready.')

In [ ]:
# =============================================================================
# Cell 5: Run all experiments (resumable within a session)
# =============================================================================
RUNS_CSV = os.path.join(OUT_DIR, 'runs.csv')
EFF_CSV = os.path.join(OUT_DIR, 'efficiency.csv')


def models_for(ds):
    if ds in CORE_DATASETS:
        return CORE_MODELS + (['ssftt'] if ds in SSFTT_DATASETS else [])
    return ['snsm']


def append_row(path, row):
    pd.DataFrame([row]).to_csv(path, mode='a', header=not os.path.exists(path), index=False)


done = set()
if os.path.exists(RUNS_CSV):
    _prev = pd.read_csv(RUNS_CSV)
    done = set(zip(_prev.dataset, _prev.model, _prev.seed))
eff_done = set()
if os.path.exists(EFF_CSV):
    _prev = pd.read_csv(EFF_CSV)
    eff_done = set(zip(_prev.dataset, _prev.model))

datasets = CORE_DATASETS + (LARGE_DATASETS if RUN_LARGE else [])
t_start = time.time()
for ds in datasets:
    todo = [(m, s) for s in SEEDS for m in models_for(ds) if (ds, m, s) not in done]
    if not todo:
        print(f'{ds}: all runs already done')
        continue
    print(f'\n=== {ds}: {len(todo)} runs ===')
    img, gt = load_dataset(ds)
    feats = preprocess(img)
    del img
    num_classes, num_bands = int(gt.max()), feats.shape[2]
    padded = pad(feats)
    np.save(os.path.join(OUT_DIR, f'_gt_{ds}.npy'), gt)

    for seed in SEEDS:
        if all((ds, m, seed) in done for m in models_for(ds)):
            continue
        tr_xy, ytr, te_xy, yte = split_centers(gt, seed)
        Xtr, Xte = extract_patches(padded, tr_xy), extract_patches(padded, te_xy)
        for m in models_for(ds):
            if (ds, m, seed) in done:
                continue
            set_seed(seed)
            model = build_model(m, num_classes, num_bands).to(device)
            r = train_and_evaluate(model, Xtr, ytr, Xte, yte, num_classes, seed)
            append_row(RUNS_CSV, {'dataset': ds, 'model': m, 'seed': seed,
                                  'n_train': len(ytr), 'n_test': len(yte),
                                  'OA': r['OA'], 'AA': r['AA'], 'kappa': r['kappa'],
                                  'fisher': r['fisher'], 'train_time_s': r['train_time_s'],
                                  'per_class': json.dumps(r['per_class'])})
            done.add((ds, m, seed))
            print(f'  seed {seed} {m:14s} OA={r["OA"]:6.2f} AA={r["AA"]:6.2f} '
                  f'k={r["kappa"]:.4f} train={r["train_time_s"]:6.1f}s '
                  f'(elapsed {(time.time() - t_start) / 60:.1f} min)')
            if (ds, m) not in eff_done:
                append_row(EFF_CSV, {'dataset': ds, 'model': m, 'bands': num_bands,
                                     **measure_efficiency(model, num_bands)})
                eff_done.add((ds, m))
            if (ds, m, seed) == (MAP_DATASET, MAP_MODEL, MAP_SEED):
                torch.save(model.state_dict(), os.path.join(OUT_DIR, f'_map_model_{ds}.pth'))
                np.save(os.path.join(OUT_DIR, f'_map_train_xy_{ds}.npy'), tr_xy)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        del Xtr, Xte
        gc.collect()
    if ds == MAP_DATASET:
        np.save(os.path.join(OUT_DIR, f'_features_{ds}.npy'), feats)
    del feats, padded
    gc.collect()

print(f'\nAll experiments finished in {(time.time() - t_start) / 60:.1f} min')

In [ ]:
# =============================================================================
# Cell 6: Paper tables (all numbers come from runs.csv / efficiency.csv)
# =============================================================================
# Mean ± sample standard deviation (ddof=1) over the 5 seeds.
runs = pd.read_csv(RUNS_CSV)
eff = pd.read_csv(EFF_CSV)
MODEL_LABELS = {'plain': 'Plain CNN', 'se_gamma': 'SE (gamma only)', 'eca_gamma': 'ECA (gamma only)', 'cbam': 'CBAM',
                'fc_gamma_beta': 'FC gamma+beta (old SSMRB)', 'snsm_gamma': 'SNSM gamma only',
                'snsm': 'SNSM gamma+beta (ours)', 'hybridsn': 'HybridSN', 'ssftt': 'SSFTT'}
MODEL_ORDER = [m for m in MODEL_LABELS if m in set(runs.model)]
DS_ORDER = [d for d in CORE_DATASETS + LARGE_DATASETS if d in set(runs.dataset)]


def fmt(vals, digits=2, scale=1.0):
    v = np.asarray(vals, dtype=float) * scale
    sd = v.std(ddof=1) if len(v) > 1 else 0.0
    return f'{v.mean():.{digits}f} ± {sd:.{digits}f}'


# Main table: OA / AA / kappa(×100) for every dataset × model
rows = []
for ds in DS_ORDER:
    for m in MODEL_ORDER:
        sub = runs[(runs.dataset == ds) & (runs.model == m)]
        if len(sub):
            rows.append({'dataset': ds, 'model': MODEL_LABELS[m], 'n_seeds': len(sub),
                         'OA': fmt(sub.OA), 'AA': fmt(sub.AA), 'kappa_x100': fmt(sub.kappa, scale=100)})
table_main = pd.DataFrame(rows)
table_main.to_csv(os.path.join(OUT_DIR, 'table_main.csv'), index=False)
print(table_main.to_string(index=False))

# Effect of beta: paired per-seed differences (same split, same seed)
pairs = [('snsm', 'snsm_gamma', 'SNSM: gamma+beta vs gamma only'),
         ('fc_gamma_beta', 'se_gamma', 'FC: gamma+beta vs gamma only'),
         ('snsm', 'fc_gamma_beta', 'SNSM gamma+beta vs FC gamma+beta'),
         ('snsm_gamma', 'eca_gamma', 'SNSM gamma only vs ECA gamma only'),
         ('snsm', 'cbam', 'SNSM gamma+beta vs CBAM'),
         ('snsm', 'plain', 'SNSM gamma+beta vs plain')]
rows = []
for ds in DS_ORDER:
    for a, b, label in pairs:
        A = runs[(runs.dataset == ds) & (runs.model == a)].set_index('seed')
        B = runs[(runs.dataset == ds) & (runs.model == b)].set_index('seed')
        common = sorted(set(A.index) & set(B.index))
        if not common:
            continue
        for metric in ['OA', 'AA']:
            d = A.loc[common, metric].values - B.loc[common, metric].values
            rows.append({'dataset': ds, 'comparison': label, 'metric': metric,
                         'mean_diff': round(float(d.mean()), 3),
                         'std_diff': round(float(d.std(ddof=1)) if len(d) > 1 else 0.0, 3),
                         'wins': f'{int((d > 0).sum())}/{len(d)}',
                         'per_seed': ' '.join(f'{x:+.2f}' for x in d)})
beta_effect = pd.DataFrame(rows)
beta_effect.to_csv(os.path.join(OUT_DIR, 'table_beta_effect.csv'), index=False)
print('\n', beta_effect.to_string(index=False))

# Per-class accuracy (mean over seeds) for the core datasets
for ds in [d for d in CORE_DATASETS if d in DS_ORDER]:
    names = CLASS_NAMES.get(ds)
    cols = {}
    for m in MODEL_ORDER:
        sub = runs[(runs.dataset == ds) & (runs.model == m)]
        if len(sub):
            arr = np.array([[np.nan if a is None else a for a in json.loads(p)] for p in sub.per_class])
            cols[MODEL_LABELS[m]] = np.round(np.nanmean(arr, axis=0), 2)
    pc = pd.DataFrame(cols)
    pc.insert(0, 'class', names if names and len(names) == len(pc) else list(range(1, len(pc) + 1)))
    gt = np.load(os.path.join(OUT_DIR, f'_gt_{ds}.npy'))
    pc.insert(1, 'labelled_pixels', [int((gt == c + 1).sum()) for c in range(len(pc))])
    pc.to_csv(os.path.join(OUT_DIR, f'table_per_class_{ds}.csv'), index=False)

# Efficiency: params / FLOPs / latency on Indian Pines input, mean training time per dataset
rows = []
for m in MODEL_ORDER:
    e = eff[eff.model == m]
    e_ip = e[e.dataset == 'IndianPines'] if 'IndianPines' in set(e.dataset) else e.head(1)
    row = {'model': MODEL_LABELS[m],
           'params_M': round(e_ip.params.iloc[0] / 1e6, 3),
           'MMACs_convlinear': None if pd.isna(e_ip.macs_convlinear.iloc[0]) else round(e_ip.macs_convlinear.iloc[0] / 1e6, 2),
           'latency_ms': round(e_ip.latency_ms.iloc[0], 2)}
    for ds in DS_ORDER:
        sub = runs[(runs.dataset == ds) & (runs.model == m)]
        if len(sub):
            row[f'train_s_{ds}'] = round(sub.train_time_s.mean(), 1)
    rows.append(row)
table_eff = pd.DataFrame(rows)
table_eff.to_csv(os.path.join(OUT_DIR, 'table_efficiency.csv'), index=False)
print('\n', table_eff.to_string(index=False))

# LaTeX for the main comparison (OA / AA), ready for the paper
core = [d for d in CORE_DATASETS if d in DS_ORDER]
lines = ['\\begin{tabular}{l' + 'cc' * len(core) + '}', '\\toprule',
         'Method & ' + ' & '.join(f'\\multicolumn{{2}}{{c}}{{{d}}}' for d in core) + ' \\\\',
         ' & ' + ' & '.join(['OA & AA'] * len(core)) + ' \\\\', '\\midrule']
for m in MODEL_ORDER:
    cells = []
    for ds in core:
        sub = runs[(runs.dataset == ds) & (runs.model == m)]
        cells += [fmt(sub.OA).replace(' ± ', '$\\pm$'), fmt(sub.AA).replace(' ± ', '$\\pm$')] if len(sub) else ['--', '--']
    lines.append(MODEL_LABELS[m].replace('gamma', '$\\gamma$').replace('beta', '$\\beta$') + ' & ' + ' & '.join(cells) + ' \\\\')
lines += ['\\bottomrule', '\\end{tabular}']
with open(os.path.join(OUT_DIR, 'table_main.tex'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print('\nWrote table_main.tex')

In [ ]:
# =============================================================================
# Cell 7: Classification map (Indian Pines, SNSM γ+β, seed 42)
# =============================================================================
# Predicts every labelled pixel; training pixels are drawn in white on the error map.
MAP_PATH = os.path.join(OUT_DIR, f'_map_model_{MAP_DATASET}.pth')
if not os.path.exists(MAP_PATH):
    print(f'No {MAP_DATASET} model in this run, skipping the map.')
else:
    feats = np.load(os.path.join(OUT_DIR, f'_features_{MAP_DATASET}.npy'))
    gt = np.load(os.path.join(OUT_DIR, f'_gt_{MAP_DATASET}.npy'))
    tr_xy = np.load(os.path.join(OUT_DIR, f'_map_train_xy_{MAP_DATASET}.npy'))
    num_classes = int(gt.max())
    model = build_model(MAP_MODEL, num_classes, feats.shape[2]).to(device)
    model.load_state_dict(torch.load(MAP_PATH, map_location=device))
    model.eval()

    coords = np.argwhere(gt > 0)
    padded = pad(feats)
    pred = np.zeros_like(gt)
    with torch.no_grad():
        for i in range(0, len(coords), 1024):
            c = coords[i:i + 1024]
            out = model(torch.from_numpy(extract_patches(padded, c)).to(device)).argmax(1).cpu().numpy()
            pred[c[:, 0], c[:, 1]] = out + 1

    train_mask = np.zeros(gt.shape, dtype=bool)
    train_mask[tr_xy[:, 0], tr_xy[:, 1]] = True
    test_mask = (gt > 0) & ~train_mask
    test_oa = (pred[test_mask] == gt[test_mask]).mean() * 100

    colors = plt.cm.tab20(np.linspace(0, 1, 20))[:num_classes]
    cmap = ListedColormap(np.vstack([[0, 0, 0, 1], colors]))
    err = np.zeros(gt.shape + (3,))
    err[test_mask & (pred == gt)] = [0.2, 0.7, 0.2]
    err[test_mask & (pred != gt)] = [0.9, 0.1, 0.1]
    err[train_mask] = [1, 1, 1]

    fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
    ax[0].imshow(gt, cmap=cmap, vmin=0, vmax=num_classes, interpolation='nearest'); ax[0].set_title('(a) Ground truth')
    ax[1].imshow(pred, cmap=cmap, vmin=0, vmax=num_classes, interpolation='nearest'); ax[1].set_title('(b) SNSM prediction')
    ax[2].imshow(err, interpolation='nearest'); ax[2].set_title(f'(c) Test errors (red), training pixels (white), OA {test_oa:.2f}%')
    for a in ax:
        a.axis('off')
    plt.tight_layout()
    for ext in ['pdf', 'png']:
        plt.savefig(os.path.join(OUT_DIR, f'classification_map_{MAP_DATASET}.{ext}'), dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved classification map (test OA on this seed: {test_oa:.2f}%)')
print(sorted(os.listdir(OUT_DIR)))